In [2]:
import pandas as pd

In [3]:
def parse_response(response):
    """
    Parses the response from an API call and extracts relevant information into a pandas Series.

    Parameters:
    response (dict): A dictionary containing the API response, which typically includes keys 
                     like 'body', 'status_code', and 'request_id'.

    Returns:
    pd.Series: A pandas Series containing the following fields:
        - status_code (int): The HTTP status code of the response.
        - request_id (str): A unique identifier for the API request.
        - completion_id (str): The ID of the completion generated by the API.
        - created (int): A Unix timestamp indicating when the response was created.
        - model (str): The model used to generate the response.
        - content (str): The content of the generated message or completion.
        - prompt_tokens (int): The number of tokens used in the prompt.
        - completion_tokens (int): The number of tokens used in the completion.
        - total_tokens (int): The total number of tokens used (prompt + completion).
        
    Notes:
    - If any of the expected keys are missing from the response, default values such as an empty dictionary or 
      empty string will be used to avoid KeyErrors.
    """
    body = response.get("body", {})
    usage = body.get("usage", {})
    choices = body.get("choices", [{}])
    message = choices[0].get("message", {}) if choices else {}

    return pd.Series({
        "status_code": response.get("status_code"),
        "request_id": response.get("request_id"),
        "completion_id": body.get("id"),
        "created": body.get("created"),
        "model": body.get("model"),
        "content": message.get("content"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
    })

In [8]:
df = pd.read_json("../results/gpt-4o-mini/wdc/2024-07-29-15-08-32.json")

In [9]:
df["task"].unique()

array(['domain-complex-free (Product)', 'domain-simple-free (Product)',
       'domain-complex-force (Product)', 'domain-simple-force (Product)'],
      dtype=object)

In [8]:
import pandas as pd
import tiktoken
from tqdm.notebook import tqdm

# Model ID constant
MODEL_ID = "gpt-4o-mini-2024-07-18"

def num_tokens_from_string(string):
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(MODEL_ID)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Read the DataFrame (assuming it's saved as CSV)
df = pd.read_json("../results/gpt-4o/baseline/batch_xriQefr3HYhJ0718awQ0MNRr_output.jsonl", lines=True)

# filter for task domain-simple-force (Product)
df = df[df["task"] == "domain-simple-force (Product)"]

# Initialize lists for token counts
input_token_counts = []
output_token_counts = []

# Process each row
for index, row in tqdm(df.iterrows(), total=len(df)):
    # Count tokens for input (chatbot_question)
    input_tokens = num_tokens_from_string(str(row['chatbot_question']))
    input_token_counts.append(input_tokens)
    
    # Count tokens for output (chatbot_response_raw)
    output_tokens = num_tokens_from_string(str(row['chatbot_response_raw']))
    output_token_counts.append(output_tokens)

# Calculate statistics
mean_input_tokens = sum(input_token_counts) / len(input_token_counts)
mean_output_tokens = sum(output_token_counts) / len(output_token_counts)

# Print results
print("\nInput Token Statistics (chatbot_question):")
print(f"Mean token count: {mean_input_tokens:.2f}")
print(f"Total number of inputs: {len(input_token_counts)}")
print(f"Min tokens: {min(input_token_counts)}")
print(f"Max tokens: {max(input_token_counts)}")

print("\nOutput Token Statistics (chatbot_response_raw):")
print(f"Mean token count: {mean_output_tokens:.2f}")
print(f"Total number of outputs: {len(output_token_counts)}")
print(f"Min tokens: {min(output_token_counts)}")
print(f"Max tokens: {max(output_token_counts)}")

# Add token counts to DataFrame
df['input_tokens'] = input_token_counts
df['output_tokens'] = output_token_counts

# Optional: Save the DataFrame with token counts
df.to_csv('dataframe_with_token_counts.csv', index=False)

KeyError: 'task'

In [4]:
df = pd.read_json("../results/gpt-4o/wdc/small-explanation/batch_j4hkMZLtMxQVGQd6I4n7NNZ4_output.jsonl", lines=True)
# filter custom_id to include wdc as a substring
df = df[df["custom_id"].str.contains("wdc-fullsize;domain-simple-force", na=False)]
# filter for custom id to include domain-simple-force (Product)
#df = df[df["custom_id"].str.contains("domain-simple-force (Product)", na=False)]
parsed_df = df["response"].apply(parse_response)

# calculate the number of prompt_tokens and completion_tokens total and mean please
print("Prompt Tokens")
print(f"Mean prompt tokens: {parsed_df['prompt_tokens'].mean()}")
print(f"Total prompt tokens: {parsed_df['prompt_tokens'].sum()}")

print("\nCompletion Tokens")
print(f"Mean completion tokens: {parsed_df['completion_tokens'].mean()}")
print(f"Total completion tokens: {parsed_df['completion_tokens'].sum()}")

Prompt Tokens
Mean prompt tokens: 75.27444444444444
Total prompt tokens: 338735

Completion Tokens
Mean completion tokens: 3.262888888888889
Total completion tokens: 14683


In [12]:
import json
import tiktoken
from tqdm import tqdm

def num_tokens_from_string(string: str, model: str = "gpt-3.5-turbo") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(model)
    num_tokens = len(encoding.encode(string))
    return num_tokens

def analyze_jsonl_tokens(file_path):
    input_tokens = []
    output_tokens = []
    
    # Read the JSONL file
    with open(file_path, 'r') as file:
        for line in tqdm(file):
            data = json.loads(line)
            
            # Count tokens for input messages
            input_text = data['messages'][0]['content']  # Assuming first message is input
            input_token_count = num_tokens_from_string(input_text)
            input_tokens.append(input_token_count)
            
            # Count tokens for output/completion
            output_text = data['messages'][1]['content']  # Assuming second message is output
            output_token_count = num_tokens_from_string(output_text)
            output_tokens.append(output_token_count)
    
    # Calculate statistics
    total_input_tokens = sum(input_tokens)
    total_output_tokens = sum(output_tokens)
    avg_input_tokens = total_input_tokens / len(input_tokens)
    avg_output_tokens = total_output_tokens / len(output_tokens)
    
    print("\nToken Analysis Results:")
    print(f"Total number of examples: {len(input_tokens)}")
    print("\nInput Tokens:")
    print(f"Total: {total_input_tokens:,}")
    print(f"Average: {avg_input_tokens:.2f}")
    print(f"Min: {min(input_tokens)}")
    print(f"Max: {max(input_tokens)}")
    
    print("\nOutput Tokens:")
    print(f"Total: {total_output_tokens:,}")
    print(f"Average: {avg_output_tokens:.2f}")
    print(f"Min: {min(output_tokens)}")
    print(f"Max: {max(output_tokens)}")
    
    print("\nCombined Tokens:")
    print(f"Total (Input + Output): {total_input_tokens + total_output_tokens:,}")
    
    return {
        'total_input_tokens': total_input_tokens,
        'total_output_tokens': total_output_tokens,
        'avg_input_tokens': avg_input_tokens,
        'avg_output_tokens': avg_output_tokens,
        'num_examples': len(input_tokens)
    }

# Run the analysis
file_path = 'wdc_train_small_structured_explanation.jsonl'
stats = analyze_jsonl_tokens(file_path)

2500it [00:00, 11300.69it/s]


Token Analysis Results:
Total number of examples: 2500

Input Tokens:
Total: 158,501
Average: 63.40
Min: 38
Max: 163

Output Tokens:
Total: 455,332
Average: 182.13
Min: 92
Max: 357

Combined Tokens:
Total (Input + Output): 613,833
